In [1]:
import pandas as pd
import numpy as np
import polars as pl
from pathlib import Path
import os
import matplotlib.pyplot as plt
import datetime

In [2]:
import pyarrow.dataset as ds

# Create a dataset object (scans the folder metadata)
dataset = ds.dataset("/home/camarada/Documents/projects/temp-grss-nasa/data_/era5_yearly_chunks", format="parquet")

# Convert to a Table or a Pandas DataFrame
df = dataset.to_table().to_pandas()

In [3]:
## cols to select
cols = ['valid_time', 'point', 'latitude', 'longitude','temp_c', 'place_id', 'name_query','capital']
dff = df[cols]

In [4]:
dff[~np.isfinite(dff['place_id'].values)]['name_query'].unique()

array(['India', 'Mexico'], dtype=object)

In [5]:
## drop all rows which contains nan values.
## This nan values are likely problems when georeferencing the city names
## this issue was propagated into the xarray ERA5 extraction.
## This basically eliminate India and Mexico from the global analysis

dff = dff.dropna().copy()

dff['month_year']=dff['valid_time'].dt.strftime("%Y-%m")

In [6]:
dff.head()

,valid_time,point,latitude,longitude,temp_c,place_id,name_query,capital,month_year
0,1981-01-01,0,42.50,1.50,-4.691660,395342932.0,Andorra,Andorra la Vella,1981-01
1,1981-01-01,1,41.25,19.75,4.475006,402962725.0,Albania,Tirana,1981-01
2,1981-01-01,2,40.25,44.50,-3.441661,193599592.0,Armenia,Yerevan,1981-01
3,1981-01-01,3,-8.75,13.25,25.745834,35017998.0,Angola,Luanda,1981-01
4,1981-01-01,4,-34.50,0.00,17.297922,409597036.0,Argentina,Buenos Aires,1981-01


In [7]:
print(dff['name_query'].value_counts())

name_query
France                  49485
Brazil                  32990
Australia               32990
Germany                 32990
Argentina               32990
                        ...  
United Arab Emirates    16495
United Kingdom          16495
Taiwan                  16495
Hong Kong               16495
Puerto Rico             16495
Name: count, Length: 109, dtype: int64


In [8]:
dff['name_query'].unique()

array(['Andorra', 'Albania', 'Armenia', 'Angola', 'Argentina', 'Austria',
       'Australia', 'Belgium', 'Bulgaria', 'Bolivia', 'Brazil', 'Belarus',
       'Canada', 'Democratic Republic of the Congo',
       'Republic of the Congo', 'Chile', "People's Republic of China",
       'Colombia', 'Costa Rica', 'Cuba', 'Cyprus', 'Czech Republic',
       'Germany', 'Denmark', 'Dominican Republic', 'Ecuador', 'Estonia',
       'Egypt', 'Ethiopia', 'Finland', 'Fiji', 'France', 'Georgia',
       'Greece', 'Guatemala', 'Hungary', 'Indonesia',
       'Republic of Ireland', 'Israel', 'Iran', 'Iceland', 'Italy',
       'Jamaica', 'Jordan', 'Japan', 'Kenya', 'Kyrgyzstan', 'South Korea',
       'Lebanon', 'Lithuania', 'Luxembourg', 'Libya', 'Madagascar',
       'Macedonia', 'Mongolia', 'Malta', 'Mauritius', 'Malaysia',
       'Mozambique', 'Namibia', 'Nigeria', 'Nicaragua',
       'Kingdom of the Netherlands', 'Norway', 'Nepal', 'New Zealand',
       'Panama', 'Peru', 'Philippines', 'Poland', 'Portugal

In [9]:
len(dff['capital'].unique())

117

In [11]:
## load NASA refence
## load nasa land-ocean temp anomaly

df_nasa = pd.read_csv('/home/camarada/Documents/projects/temp-grss-nasa/data_/nasa/1981-2025.csv')

df_long = df_nasa.melt(id_vars=['Year'], var_name='Month', value_name='Value')

df_long['Date'] = pd.to_datetime(df_long['Year'].astype(str) + df_long['Month'], format='%Y%b')

# 3. Clean up: Sort by date and set it as the index
df_ts = df_long.sort_values('Date').set_index('Date').drop(columns=['Year', 'Month'])

df_ts.reset_index(inplace=True)

## create a new column YEAR-MONTH
df_ts['month_year']=pd.to_datetime(df_ts['Date']).dt.strftime("%Y-%m")

## apply scaler to celsius
df_ts['Value'] = df_ts['Value']/ 100


df_ts = df_ts.dropna(axis=0)

In [12]:
### Create arrays and save as npz

## root_dir =
root_dir = os.getcwd()
n_capitals = 117  # number of capitals
n_timepoints = 31  # days


## preprocessing SORT
## turn in lower all capitals to sort it properly
dff['capital'] = dff['capital'].apply(lambda x:x.lower())
dff = dff.sort_values(by=['valid_time','capital']).copy()

uniq_capital = dff['capital'].unique()
month_years = dff['month_year'].unique()
n_samples = len(month_years)

# Pre-allocate arrays
temp_matrices = np.zeros((n_samples, n_capitals, n_timepoints), dtype=np.float32)
masks = np.zeros((n_samples, n_capitals, n_timepoints), dtype=np.float32)
labels = np.zeros(n_samples, dtype=np.float32)
month_year_list = []

for idx, my in enumerate(month_years):
    print(f"processing:{my}")
    # Get data for this month/year
    month_data = dff[dff['month_year'] == my]

    for i, capital in enumerate(uniq_capital):
        capital_data = month_data[month_data['capital'] == capital].sort_values('valid_time')
        temp_values = capital_data['temp_c'].values

        if len(temp_values) > 0:
            temp_matrices[idx, i, :len(temp_values)] = temp_values
            masks[idx, i, :len(temp_values)] = 1.0

    # Get the true anomaly
    anomaly_value = df_ts[df_ts['month_year'] == my]['Value'].values[0]
    labels[idx] = anomaly_value
    month_year_list.append(my)



processing:1981-01
processing:1981-02
processing:1981-03
processing:1981-04
processing:1981-05
processing:1981-06
processing:1981-07
processing:1981-08
processing:1981-09
processing:1981-10
processing:1981-11
processing:1981-12
processing:1982-01
processing:1982-02
processing:1982-03
processing:1982-04
processing:1982-05
processing:1982-06
processing:1982-07
processing:1982-08
processing:1982-09
processing:1982-10
processing:1982-11
processing:1982-12
processing:1983-01
processing:1983-02
processing:1983-03
processing:1983-04
processing:1983-05
processing:1983-06
processing:1983-07
processing:1983-08
processing:1983-09
processing:1983-10
processing:1983-11
processing:1983-12
processing:1984-01
processing:1984-02
processing:1984-03
processing:1984-04
processing:1984-05
processing:1984-06
processing:1984-07
processing:1984-08
processing:1984-09
processing:1984-10
processing:1984-11
processing:1984-12
processing:1985-01
processing:1985-02
processing:1985-03
processing:1985-04
processing:1

In [13]:
# Save to npz file
path_output = '/home/camarada/Documents/projects/temp-grss-nasa/data_/tensors_ready/NASA-TEMP-LAND-OCEAR'
os.makedirs(path_output, exist_ok = True) 
np.savez_compressed(
    os.path.join(path_output, 'era5-1981-2026fev-temp-clean.npz'),
    temp_matrices=temp_matrices,  # shape: (n_samples, n_capitals, n_timepoints)
    masks=masks,                  # shape: (n_samples, n_capitals, n_timepoints)
    labels=labels,                # shape: (n_samples,)
    month_years=np.array(month_year_list, dtype='U'),  # shape: (n_samples,)
    capitals=np.array(uniq_capital, dtype='U')         # shape: (n_capitals,)
)